# Fish Toxicty Regression

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import PredefinedSplit



df = pd.read_csv(
    "../adore_dataset/processed/s-F2F-1_mortality.csv",
    dtype=str
)

split_def = df["split_occurrence"]



starts_with_tax = [
    col for col in df.columns
    if col.startswith("tax")
]


starts_with_split = [
    col for col in df.columns
    if col.startswith("split")
]



cols_to_drop = [

    # other LC50 results
    # keep result_conc1_mean_binary for classification
    # keep result_conc1_mean_mol_log for regression
    "result_conc1_mean",
    "result_conc1_mean_mol",
    "result_conc1_mean_log",
    "result_conc1_mean_op",

    # IDs and names
    "test_id",
    "result_id",
    "reference_number",
    "test_cas",
    "test_cas_name",
    "chem_name",
    "chem_dtxsid",
    "chem_dtxcid",
    "chem_inchi",
    "chem_inchikey",
    "chem_pcp_cid",
    "chem_pcp_inchikey",
    "chem_pcp_inchi",
    "chem_pcp_iupac_name",
    "chem_pcp_can_smiles",
    "chem_rdkit_can_smiles",
    "chem_sf",

    # fingerprints
    "chem_pcp_fp",
    "chem_MACCS_fp",
    "chem_Morgan_fp",
    "chem_ToxPrint_fp",

    # not useful
    "test_location",
    "test_control_type",
    "test_application_freq_unit",
    "test_organism_lifestage",
    "result_effect",
    "result_endpoint",
    "result_conc1_type",
    "test_media_type",
    "chem_ws_binary",
    "chem_mol2vec_allowed",
    "media_ph_mean",
    "media_temperature_mean",
    "chem_pka_median",
    "species_number"
]


# add tax and split columns to columns to drop
cols_to_drop.extend(starts_with_tax)
cols_to_drop.extend(starts_with_split)


print("Original shape:", df.shape)

df = df.drop(
    columns=cols_to_drop,
    errors="ignore"
)

# add split_occurrence 
df["split_occurrence"] = split_def

print("Cleaned shape:", df.shape)


# define cols

numeric_cols = []

for col in df.columns:

    converted = pd.to_numeric(
        df[col],
        errors="coerce"
    )

    if converted.notna().sum() == df[col].notna().sum():
        numeric_cols.append(col)


categorical_cols = [
    col for col in df.columns
    if col not in numeric_cols
]


Original shape: (4641, 1106)
Cleaned shape: (4641, 1034)


In [2]:
# test and train sets
test_set = df[
    df["split_occurrence"] == "test"
].copy()

train_set = df[
    df["split_occurrence"] != "test"
].copy()



# define split

test_fold = (
    train_set["split_occurrence"]
    .astype(int)
    .values
)

ps = PredefinedSplit(test_fold)

In [5]:


# define X and y for regression

X_train_reg = train_set.drop(
    columns=[
        "split_occurrence",
        "result_conc1_mean_binary",
        "result_conc1_mean_mol_log",
        "test_exposure_type"
    ]
)

y_train_reg = pd.to_numeric(
    train_set["result_conc1_mean_mol_log"],
    errors="coerce"
)


X_test_reg = test_set.drop(
    columns=[
        "split_occurrence",
        "result_conc1_mean_binary",
        "result_conc1_mean_mol_log",
        "test_exposure_type"
    ]
)

y_test_reg = pd.to_numeric(
    test_set["result_conc1_mean_mol_log"],
    errors="coerce"
)
for col in X_train_reg.columns:
    converted = pd.to_numeric(X_train_reg[col], errors="coerce")

    if converted.notna().mean() > 0.9:
        X_train_reg[col] = converted
        X_test_reg[col] = pd.to_numeric(
            X_test_reg[col],
            errors="coerce"
        )

numeric_cols = X_train_reg.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = X_train_reg.select_dtypes(
    include=["object", "category"]
).columns.tolist()

print("Regression X train:", X_train_reg.shape)
print("Regression y train:", y_train_reg.shape)

print("Regression X test:", X_test_reg.shape)
print("Regression y test:", y_test_reg.shape)

numeric_cols

Regression X train: (3928, 1030)
Regression y train: (3928,)
Regression X test: (713, 1030)
Regression y test: (713,)


['result_obs_duration_mean',
 'chem_mw',
 'chem_mp',
 'chem_ws',
 'chem_pcp_heavy_atom_count',
 'chem_rdkit_clogp',
 'chem_pcp_bonds_count',
 'chem_pcp_doublebonds_count',
 'chem_pcp_triplebonds_count',
 'chem_rings_count',
 'chem_OH_count',
 'chem_mol2vec000',
 'chem_mol2vec001',
 'chem_mol2vec002',
 'chem_mol2vec003',
 'chem_mol2vec004',
 'chem_mol2vec005',
 'chem_mol2vec006',
 'chem_mol2vec007',
 'chem_mol2vec008',
 'chem_mol2vec009',
 'chem_mol2vec010',
 'chem_mol2vec011',
 'chem_mol2vec012',
 'chem_mol2vec013',
 'chem_mol2vec014',
 'chem_mol2vec015',
 'chem_mol2vec016',
 'chem_mol2vec017',
 'chem_mol2vec018',
 'chem_mol2vec019',
 'chem_mol2vec020',
 'chem_mol2vec021',
 'chem_mol2vec022',
 'chem_mol2vec023',
 'chem_mol2vec024',
 'chem_mol2vec025',
 'chem_mol2vec026',
 'chem_mol2vec027',
 'chem_mol2vec028',
 'chem_mol2vec029',
 'chem_mol2vec030',
 'chem_mol2vec031',
 'chem_mol2vec032',
 'chem_mol2vec033',
 'chem_mol2vec034',
 'chem_mol2vec035',
 'chem_mol2vec036',
 'chem_mol2vec037'

In [5]:
# regression models
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor,
    GradientBoostingRegressor,
    AdaBoostRegressor,
    ExtraTreesRegressor
)
from sklearn.neural_network import MLPRegressor

In [6]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(),
    "Lasso": Lasso(),
    "Elastic Net": ElasticNet(),
    "KNN": KNeighborsRegressor(),
    "SVR": SVR(),
    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(random_state=42),
    "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    "AdaBoost": AdaBoostRegressor(random_state=42),
    "Extra Trees": ExtraTreesRegressor(random_state=42),
    "MLP": MLPRegressor(max_iter=2000, random_state=42)
}

In [ ]:
# pre-processing and cross validation
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import cross_val_score


preprocess = ColumnTransformer([
    ("standard-scaler", StandardScaler(), numeric_cols),
    ("one-hot-encoder", OneHotEncoder(handle_unknown="ignore"), categorical_cols)
])

for name, model in models.items():
    model_reg = make_pipeline(preprocess,model)
    score = cross_val_score(model_reg,X_train_reg,y_train_reg,cv=ps,scoring="r2")
    print(name,score.mean())


Linear Regression 0.4622882263763878
Ridge 0.4622709674190091
Lasso -0.023094719175950917
Elastic Net -0.023094719175950917
KNN 0.3846997402904216
SVR 0.25969531710202187
Decision Tree 0.2483241746634061
